In [11]:
import os
os.chdir('/Users/yaminiumarani/Desktop/rag_thesis_project')
print(os.getcwd())

/Users/yaminiumarani/Desktop/rag_thesis_project


In [12]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load clean dataset
df = pd.read_csv('data/github_issues_final.csv', low_memory=False)
print("Dataset loaded:", df.shape)

# Load embedding model
print("\nLoading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model ready! ✅")

Dataset loaded: (15955, 37)

Loading embedding model...


Task was destroyed but it is pending!
task: <Task pending name='Task-511' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/yaminiumarani/anaconda3/envs/rag_thesis/lib/python3.11/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-512' coro=<Kernel.shell_main() running at /Users/yaminiumarani/anaconda3/envs/rag_thesis/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/yaminiumarani/anaconda3/envs/rag_thesis/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py:563]>
/Users/yaminiumarani/anaconda3/envs/rag_thesis/lib/python3.11/site-packages/h11/_headers.py:205: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  new_headers.append((raw_name, name, value))
Task was destroyed but it is pending!
task: <Task pending name='Task-512' coro=<Kernel.shell_main() running at /Users/yaminiumarani/anaconda3/envs/rag_thesis/lib/python3.11/site-package

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model ready! ✅


In [3]:
# Create embeddings for all issues

print("Creating embeddings for all issues...")
print("This will take a few minutes, please wait...\n")

texts = df['combined_text'].tolist()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\nEmbeddings created! ✅")
print(f"Shape: {embeddings.shape}")
print(f"Each issue is now a vector of {embeddings.shape[1]} numbers")

Creating embeddings for all issues...
This will take a few minutes, please wait...



Batches:   0%|          | 0/250 [00:00<?, ?it/s]


Embeddings created! ✅
Shape: (15955, 384)
Each issue is now a vector of 384 numbers


In [4]:
# Build FAISS index
print("Building FAISS index...")

# Get the dimension of our embeddings
dimension = embeddings.shape[1]

# Create a flat L2 index (simple baseline - no tricks)
index = faiss.IndexFlatL2(dimension)

# Add all embeddings to the index
index.add(embeddings.astype(np.float32))

print(f"FAISS index built! ✅")
print(f"Total vectors indexed: {index.ntotal}")

Building FAISS index...
FAISS index built! ✅
Total vectors indexed: 15955


In [5]:
# Save the FAISS index and embeddings to disk

import os

# Save FAISS index
faiss.write_index(index, 'data/baseline_faiss.index')

# Save embeddings as numpy file
np.save('data/baseline_embeddings.npy', embeddings)

print("FAISS index saved! ✅")
print("Embeddings saved! ✅")
print("\nFiles saved:")
print(f"  - data/baseline_faiss.index ({os.path.getsize('data/baseline_faiss.index') / 1024 / 1024:.1f} MB)")
print(f"  - data/baseline_embeddings.npy ({os.path.getsize('data/baseline_embeddings.npy') / 1024 / 1024:.1f} MB)")

FAISS index saved! ✅
Embeddings saved! ✅

Files saved:
  - data/baseline_faiss.index (23.4 MB)
  - data/baseline_embeddings.npy (23.4 MB)


In [13]:
index = faiss.read_index('data/baseline_faiss.index')
print("Index loaded:", index.ntotal, "vectors")

Index loaded: 15955 vectors


In [14]:
# Build a simple retrieval function

def retrieve_similar_issues(query, top_k=5):
    # Embed the query
    query_embedding = model.encode([query], convert_to_numpy=True)
    
    # Search the FAISS index
    distances, indices = index.search(query_embedding.astype(np.float32), top_k)
    
    # Return the top results
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            'rank': i + 1,
            'distance': distances[0][i],
            'title': df['title'].iloc[idx],
            'label': df['labels_0_name'].iloc[idx],
            'author': df['answers_0_author'].iloc[idx],
            'combined_text': str(df['combined_text'].iloc[idx])
        })
    return results

# Test it with a real query
query = "window resize not working correctly"
print(f"Query: '{query}'")
print("="*60)

results = retrieve_similar_issues(query, top_k=5)
for r in results:
    print(f"\nRank {r['rank']} (distance: {r['distance']:.4f})")
    print(f"Title: {r['title']}")
    print(f"Label: {r['label']}")
    print(f"Preview: {r['combined_text'][:200]}")
    print("-"*40)

Query: 'window resize not working correctly'

Rank 1 (distance: 0.7838)
Title: [Feature Request] Allow resizeTo() (and other resize functionality) to work on windows where "resizable" is false
Label: bug
Preview: TITLE: [Feature Request] Allow resizeTo() (and other resize functionality) to work on windows where "resizable" is false
LABEL: bug
BODY: NWJS Version : 0.52.0 Operating System : Windows 10 (I assume 
----------------------------------------

Rank 2 (distance: 0.8378)
Title: Rectangle Doesn't Resize Windows
Label: bug
Preview: TITLE: Rectangle Doesn't Resize Windows
LABEL: bug
BODY: macOS Version: 12.6.2 Rectangle Version: 0.64 I've given Rectangle the requested permissions, and to no avail does it resize my windows. I'm on
----------------------------------------

Rank 3 (distance: 0.8405)
Title: Dragging windows between screens makes them resize
Label: bug
Preview: TITLE: Dragging windows between screens makes them resize
LABEL: bug
BODY: - [x] This is a new bug that hasn't 

In [15]:
# Test with two more queries

queries = [
    "authentication token expiring too quickly",
    "dark mode not applying to all components"
]

for query in queries:
    print(f"\nQuery: '{query}'")
    print("="*60)
    results = retrieve_similar_issues(query, top_k=3)
    for r in results:
        print(f"Rank {r['rank']} | Distance: {r['distance']:.4f}")
        print(f"Title: {r['title']}")
        print(f"Label: {r['label']}")
        print("-"*40)


Query: 'authentication token expiring too quickly'
Rank 1 | Distance: 0.8112
Title: Clarify cookie session refresh/invalidation
Label: bug
----------------------------------------
Rank 2 | Distance: 0.8608
Title: Token-refresh failing for performance tests
Label: bug
----------------------------------------
Rank 3 | Distance: 0.9160
Title: getIdToken throws "auth/user-token-expired" if refresh token is revoked
Label: api: auth
----------------------------------------

Query: 'dark mode not applying to all components'
Rank 1 | Distance: 0.8357
Title: [Blazor] Some of the default UI styling is not accessible in dark mode
Label: bug
----------------------------------------
Rank 2 | Distance: 0.8971
Title: Deployed links sometimes start in dark mode
Label: bug
----------------------------------------
Rank 3 | Distance: 0.9037
Title: [Documentation] v-hover example not good in dark mode
Label: good first issue
----------------------------------------


Migrating to Ollama generation model from GPT-3.5 Turbo to ensure consistency with other configurations (dynamic weighted and partition dynamic weighted) and to account for token quota and limitations

In [16]:
import ollama

GEN_OPTIONS = {'temperature': 0, 'num_ctx': 4096}

def build_context(results):
    context = ""
    for i, r in enumerate(results):
        context += f"\n--- Issue {i+1} ---\n"
        context += f"Title: {r['title']}\n"
        context += f"Label: {r['label']}\n"
        context += f"Details: {r['combined_text']}\n"
    return context

def build_prompt(query, context):
    return f"""You are an IT support assistant. Based on similar resolved issues below, give a concise resolution recommendation for the new issue.

NEW ISSUE: {query}

SIMILAR ISSUES:
{context}

Give a short, specific recommendation in 3-4 sentences."""

def baseline_rag(query, top_k=3):
    results = retrieve_similar_issues(query, top_k=top_k)
    context = build_context(results)
    response = ollama.chat(
        model='llama3.2:1b',
        options=GEN_OPTIONS,
        messages=[{'role': 'user', 'content': build_prompt(query, context)}]
    )
    return {
        'query': query,
        'retrieved_issues': results,
        'recommendation': response['message']['content']
    }

out = baseline_rag("window resize not working correctly")
print(out['recommendation'])

Based on similar resolved issues, here's a concise resolution recommendation for the new issue:

To resolve the "window resize not working correctly" issue where `win.resizeTo()` or other NW.js API calls fail to resize windows when set as non-resizable (`resizable: false`), I recommend adding an additional check in your code to ensure that the window is not already being resized before attempting to resize it. This can be achieved by checking if the window's current size is different from its initial size, and only then attempt to resize it. Additionally, you may want to consider implementing a "resize timeout" mechanism to prevent excessive resizing attempts when the window is already in a fixed position or size.


In [17]:
import json

test_queries = [
    "window resize not working correctly",
    "authentication token expiring too quickly",
    "dark mode not applying to all components"
]

baseline_results = {
    'configuration': 'baseline',
    'generation_model': 'llama3.2:1b',
    'retrieval': 'flat index, semantic similarity only',
    'results': []
}

for query in test_queries:
    print(f"Running: {query}")
    output = baseline_rag(query)
    baseline_results['results'].append({
        'query': output['query'],
        'retrieved_titles': [r['title'] for r in output['retrieved_issues']],
        'recommendation': output['recommendation']
    })

with open('results/baseline_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

print("\nSaved with model provenance ✅")

Running: window resize not working correctly
Running: authentication token expiring too quickly
Running: dark mode not applying to all components

Saved with model provenance ✅


In [10]:
import pandas as pd
old = pd.read_csv('_pre_budget/data/github_issues_final.csv', low_memory=False)
new = pd.read_csv('data/github_issues_final.csv', low_memory=False)

cols = [c for c in new.columns if c != 'combined_text']
print("same shape:", old.shape == new.shape)
print("non-text columns identical:", old[cols].equals(new[cols]))
print("combined_text changed:", not old['combined_text'].equals(new['combined_text']))

Task was destroyed but it is pending!
task: <Task pending name='Task-80' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/yaminiumarani/anaconda3/envs/rag_thesis/lib/python3.11/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-81' coro=<Kernel.shell_main() running at /Users/yaminiumarani/anaconda3/envs/rag_thesis/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/yaminiumarani/anaconda3/envs/rag_thesis/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py:563]>
/Users/yaminiumarani/anaconda3/envs/rag_thesis/lib/python3.11/site-packages/pandas/core/internals/managers.py:2430: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  ensure_block_shape(arr, ndim=2), placement=BlockPlacement(i), refs=ref
Task was destroyed but it is pending!
task: <Task pending name='Task-81' coro=<Kernel.shell_main() running at /Users/yaminiumarani/anaconda3/

same shape: True
non-text columns identical: True
combined_text changed: True


In [18]:
results = retrieve_similar_issues("window resize not working correctly", top_k=3)
for r in results:
    print("="*70)
    print(r['title'])
    print("-"*70)
    print(r['combined_text'])
    print()

[Feature Request] Allow resizeTo() (and other resize functionality) to work on windows where "resizable" is false
----------------------------------------------------------------------
TITLE: [Feature Request] Allow resizeTo() (and other resize functionality) to work on windows where "resizable" is false
LABEL: bug
BODY: NWJS Version : 0.52.0 Operating System : Windows 10 (I assume also Linux) ### Expected behavior When using `win.resizeTo()` (or any other NW.js API calls that resize windows) on a window that's set as `resizable: false`, it should still resize the window. ### Actual behavior The window is not resized. If `win.setResizable(true)` is called before calling `win.resizeTo()`, the window will be properly
REPLY_1 by ayushmanchhabra: Isn't the point of `resizable: false` for not to be able to resize the window?
REPLY_2 by sysrage: > Isn't the point of `resizable: true` for not to be able to resize the window? I suppose that depends on your opinion! Personally, I believe `resiz

In [19]:
import json
import pandas as pd

df = pd.read_csv('data/github_issues_final.csv', low_memory=False)

checks = {
    "dark mode not applying to all components": [
        'color-scheme', 'light only', 'reconnect', 'error boundary'],
    "authentication token expiring too quickly": [
        'Cookie expire', 'Cookie refresh', 'revoked', '15 seconds'],
}

for q, phrases in checks.items():
    texts = [r['combined_text']
             for r in retrieve_similar_issues(q, top_k=3)]
    print(f"\n{q}")
    for p in phrases:
        print(f"  {p!r:22} present: {any(p.lower() in t.lower() for t in texts)}")


dark mode not applying to all components
  'color-scheme'         present: True
  'light only'           present: True
  'reconnect'            present: True
  'error boundary'       present: True

authentication token expiring too quickly
  'Cookie expire'        present: True
  'Cookie refresh'       present: True
  'revoked'              present: True
  '15 seconds'           present: True


In [20]:
for r in retrieve_similar_issues("dark mode not applying to all components", top_k=3):
    print("=" * 78)
    print(r['title'])
    print("-" * 78)
    print(r['combined_text'])
    print()

[Blazor] Some of the default UI styling is not accessible in dark mode
------------------------------------------------------------------------------
TITLE: [Blazor] Some of the default UI styling is not accessible in dark mode
LABEL: bug
BODY: There are cases where the styling of some of Blazor's built-in UI elements gets adversely affected by global dark mode styling. Some examples: 1. The server reconnect UI: ![Blazor server reconnect UI]( 2. The "Yellow bar": ![Yellow bar]( We should consider styling these elements in ways that don't inherit from other styles defined in the app.
REPLY_1 by javiercn: Looked at the four pieces of UI we produce: * Blazor Error UI (yellow bar) needs `color-scheme: light only` on the template to avoid issues with auto dark mode. * Blazor Loading animation: Seems fine, no changes needed. * Error boundary UI: It's fine, no changes needed. * Blazor Server reconnect UI. From what was described in this issue, there were problems when styles where changes. To